In [ ]:
import pandas as pd
import numpy as np
from ctgan import CTGAN
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# Définition des variations permises
variations = {
    'Population': 0.01,  # ±1%
    'Lignes de bus': 0.05,  # ±5%
    'Petits taxis': 0.07,  # ±7%
    'Grands taxis': 0.07,  # ±7%
    'Nombre de tramways': 0.03,  # ±3%
    'Nombre de gares ferroviaires': 0,  # Pas de variation
    'Nombre de stations de tram': 0.05,  # ±5%
    'Points de chargement des grands taxis': 0.10,  # ±10%
    'Arrêts de bus': 0.08,  # ±8%
    'Passagers/jour (bus)': 0.10,  # ±10%
    'Passagers/jour (trains)': 0.07,  # ±7%
    'Passagers/jour (tramways)': 0.07,  # ±7%
}

# Données originales
data = {
    'Régions': [
        'Tanger-Tétouan-Al Hoceïma', 'Rabat-Salé-Kénitra', 'Casablanca-Settat',
        'Marrakech-Safi', 'Fès-Meknès'
    ],
    'Population': [4030222, 5132639, 7688967, 4892393, 4467911],
    'Lignes de bus': [145, 137, 168, 92, 52],
    'Petits taxis': [1450, 1200, 7000, 850, 2000],
    'Grands taxis': [1500, 2000, 7500, 1500, 1000],
    'Nombre de tramways': [0, 60, 45, 0, 0],
    'Nombre de gares ferroviaires': [9, 7, 24, 8, 12],
    'Nombre de stations de tram': [0, 21, 70, 0, 0],
    'Points de chargement des grands taxis': [80, 40, 90, 30, 120],
    'Arrêts de bus': [946, 1100, 1500, 1700, 400],
    'Passagers/jour (bus)': [300000, 240000, 140000, 297000, 244000],
    'Passagers/jour (trains)': [11000, 15000, 180000, 9000, 15000],
    'Passagers/jour (tramways)': [0, 200000, 300000, 0, 0]
}

def appliquer_variations_et_contraintes(row, data_orig, variations):
    """
    Applique à la fois les variations et les contraintes à une ligne de données
    """
    region = row['Régions']
    orig_values = data_orig[data_orig['Régions'] == region].iloc[0]
    
    # Appliquer les variations
    for col, variation in variations.items():
        if col != 'Régions':
            orig_val = orig_values[col]
            if orig_val != 0:
                max_variation = orig_val * variation
                variation_value = np.random.normal(0, max_variation/2)
                new_val = orig_val + variation_value
                # Garantir que les valeurs restent dans les limites
                lower_bound = max(0, orig_val - max_variation)
                upper_bound = orig_val + max_variation
                row[col] = np.clip(new_val, lower_bound, upper_bound)
            else:
                row[col] = 0
    
    # Appliquer les contraintes spécifiques aux tramways
    if region not in ['Casablanca-Settat', 'Rabat-Salé-Kénitra']:
        row['Nombre de tramways'] = 0
        row['Nombre de stations de tram'] = 0
        row['Passagers/jour (tramways)'] = 0
    else:
        row['Nombre de tramways'] = np.clip(row['Nombre de tramways'], 30, 70)
        row['Nombre de stations de tram'] = np.clip(row['Nombre de stations de tram'], 20, 80)
        row['Passagers/jour (tramways)'] = np.clip(row['Passagers/jour (tramways)'], 150000, 350000)
    
    return row

def prepare_training_data(df_original):
    """
    Prépare les données d'entraînement en créant plusieurs versions modifiées
    """
    expanded_data = []
    
    for _, row in df_original.iterrows():
        # Ajouter la ligne originale
        expanded_data.append(row)
        # Créer 5 copies modifiées
        for _ in range(5):
            new_row = appliquer_variations_et_contraintes(row.copy(), df_original, variations)
            expanded_data.append(new_row)
    
    expanded_df = pd.DataFrame(expanded_data)
    
    # Arrondir les valeurs numériques
    numeric_columns = expanded_df.select_dtypes(include=[np.number]).columns
    expanded_df[numeric_columns] = expanded_df[numeric_columns].round(0)
    
    return expanded_df

# Création du DataFrame initial et préparation des données étendues
df = pd.DataFrame(data)
expanded_df = prepare_training_data(df)

# Préparation pour CTGAN
scaler = MinMaxScaler()
numeric_columns = expanded_df.select_dtypes(include=[np.number]).columns
df_scaled = expanded_df.copy()
df_scaled[numeric_columns] = scaler.fit_transform(expanded_df[numeric_columns])

# Configuration et entraînement du CTGAN
ctgan = CTGAN(
    epochs=300,
    batch_size=50,
    generator_dim=(256, 256),
    discriminator_dim=(256, 256),
    verbose=True
)
discrete_columns = ['Régions']
ctgan.fit(df_scaled, discrete_columns)

# Génération des données synthétiques
synthetic_data = []
samples_per_region = 20

for region in data['Régions']:
    temp_samples = []
    for _ in range(samples_per_region):
        # Générer un échantillon
        sample = ctgan.sample(1)
        sample['Régions'] = region
        # Inverse transformer les données
        sample[numeric_columns] = scaler.inverse_transform(sample[numeric_columns])
        # Appliquer les variations et contraintes
        sample = appliquer_variations_et_contraintes(sample.iloc[0], df, variations)
        temp_samples.append(sample)
    
    region_samples = pd.DataFrame(temp_samples)
    synthetic_data.append(region_samples)

# Post-traitement
synthetic_df = pd.concat(synthetic_data, ignore_index=True)

# Nettoyage final
synthetic_df = synthetic_df.drop_duplicates()
numeric_columns = synthetic_df.select_dtypes(include=[np.number]).columns
synthetic_df[numeric_columns] = synthetic_df[numeric_columns].round(0)
synthetic_df[numeric_columns] = synthetic_df[numeric_columns].abs()

# Sauvegarde des données
expanded_df.to_csv('donnees_etendues.csv', index=False)
synthetic_df.to_csv('donnees_synthetiques_finales.csv', index=False)

# Affichage des statistiques
print("\nStatistiques des données étendues:")
print(f"Nombre de lignes dans les données étendues: {len(expanded_df)}")
print("\nStatistiques des données synthétiques finales:")
print(f"Nombre total d'échantillons: {len(synthetic_df)}")
print("\nDistribution des régions:")
print(synthetic_df['Régions'].value_counts())

# Vérification des contraintes
print("\nVérification des contraintes sur les tramways:")
print(synthetic_df.groupby('Régions')[['Nombre de tramways', 'Nombre de stations de tram', 'Passagers/jour (tramways)']].mean())

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mutual_info_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def evaluate_synthetic_data(real_df, synthetic_df, numeric_columns):
    """
    Comprehensive evaluation of synthetic data quality
    """
    evaluation_results = {}
    
    # 1. Statistical Distribution Tests (Kolmogorov-Smirnov)
    ks_test_results = {}
    for col in numeric_columns:
        statistic, p_value = ks_2samp(real_df[col], synthetic_df[col])
        ks_test_results[col] = {
            'statistic': statistic,
            'p_value': p_value
        }
    evaluation_results['ks_tests'] = ks_test_results
    
    # 2. Calculate correlation difference
    real_corr = real_df[numeric_columns].corr()
    synthetic_corr = synthetic_df[numeric_columns].corr()
    correlation_diff = np.abs(real_corr - synthetic_corr)
    evaluation_results['correlation_diff'] = correlation_diff
      
    # 3. Privacy Metrics - Nearest Neighbor Distance Ratio
    def calculate_nearest_neighbor_distance(df):
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df[numeric_columns])
        distances = []
        for i in range(len(scaled_data)):
            dist = np.sqrt(np.sum((scaled_data - scaled_data[i])**2, axis=1))
            dist.sort()
            distances.append(dist[1])  # Second smallest distance (first is 0 - distance to self)
        return np.mean(distances)
    
    real_nn_dist = calculate_nearest_neighbor_distance(real_df)
    synthetic_nn_dist = calculate_nearest_neighbor_distance(synthetic_df)
    privacy_score = synthetic_nn_dist / real_nn_dist
    evaluation_results['privacy_score'] = privacy_score
    
    # 4. Column-wise statistics comparison
    stats_comparison = pd.DataFrame()
    for col in numeric_columns:
        real_stats = real_df[col].describe()
        synth_stats = synthetic_df[col].describe()
        stats_comparison[f'{col}_real'] = real_stats
        stats_comparison[f'{col}_synthetic'] = synth_stats
    evaluation_results['stats_comparison'] = stats_comparison
    
    return evaluation_results

def plot_evaluation_results(real_df, synthetic_df, numeric_columns, evaluation_results):
    """
    Create visualizations for evaluation results
    """
    # 1. Distribution plots


    rows, cols = 4, 3  # Définissez le nombre de lignes et de colonnes souhaité
    fig, axes = plt.subplots(rows, cols, figsize=(15, 10))
    
    # Aplatir les axes pour itérer facilement
    axes = axes.flatten()
    
    for i, col in enumerate(numeric_columns):
        if i < len(axes):  # Assurez-vous de ne pas dépasser le nombre d'axes disponibles
            sns.kdeplot(data=real_df[col], label='Real', alpha=0.5, ax=axes[i])
            sns.kdeplot(data=synthetic_df[col], label='Synthetic', alpha=0.5, ax=axes[i])
            axes[i].set_title(f'Distribution Comparison - {col}')
            axes[i].legend()
    
    # Supprimez les sous-graphiques vides si le nombre de graphiques est inférieur à rows * cols
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    plt.tight_layout()  # Ajuste l'espacement entre les sous-graphiques
    plt.show()
    
    # 2. Correlation heatmap comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
    
    sns.heatmap(evaluation_results['correlation_diff'], 
                annot=True, cmap='coolwarm', center=0,
                ax=ax1)
    ax1.set_title('Correlation Difference (Synthetic - Real)')
    
    # 3. PCA visualization
    scaler = StandardScaler()
    pca = PCA(n_components=2)
    
    real_scaled = scaler.fit_transform(real_df[numeric_columns])
    synthetic_scaled = scaler.transform(synthetic_df[numeric_columns])
    
    real_pca = pca.fit_transform(real_scaled)
    synthetic_pca = pca.transform(synthetic_scaled)
    
    sns.scatterplot(x=real_pca[:, 0], y=real_pca[:, 1], 
                    label='Real', alpha=0.5, ax=ax2)
    sns.scatterplot(x=synthetic_pca[:, 0], y=synthetic_pca[:, 1], 
                    label='Synthetic', alpha=0.5, ax=ax2)
    ax2.set_title('PCA Visualization')
    ax2.legend()
    
    plt.tight_layout()
    
    plt.show()

# Run evaluation
numeric_columns = df.select_dtypes(include=[np.number]).columns
evaluation_results = evaluate_synthetic_data(df, synthetic_df, numeric_columns)

# Print evaluation results
print("\nKolmogorov-Smirnov Test Results:")
for col, results in evaluation_results['ks_tests'].items():
    print(f"\n{col}:")
    print(f"Statistic: {results['statistic']:.4f}")
    print(f"P-value: {results['p_value']:.4f}")

print("\nPrivacy Score (Nearest Neighbor Distance Ratio):")
print(f"Score: {evaluation_results['privacy_score']:.4f}")
print("(Score > 1 indicates good privacy preservation)")

print("\nColumn-wise Statistics Comparison:")
print(evaluation_results['stats_comparison'])

# Create visualizations
plot_evaluation_results(df, synthetic_df, numeric_columns, evaluation_results)
    